# Optical flow

Optical flow estimates how every pixel moves between two images. For each pixel `(x, y)` of the first image, the flow `flow[y, x] = (dx, dy)` tells where that pixel ended up in the second image.

Unlike registration, which fits a single global transformation (a homography) to the whole image, optical flow is **dense**: each pixel gets its own displacement, so different parts of the scene can move differently, e.g. a moving object over a still background.

Available methods:

- **FARNEBACK**: classical polynomial-expansion algorithm from OpenCV, fast and model-free.
- **RAFT_SMALL** / **RAFT_LARGE**: deep-learning models from torchvision, more robust on large motions and weak textures.

Optical flow works best on **small motions**, like consecutive video frames. For photos taken from very different viewpoints, align them first with the `RegistrationPipeline` (see the registration notebook).

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from visionpipelines import OpticalFlowPipeline, OpticalFlowMethod

## A simple example

To check the result we create two frames where we know exactly how things move: a checkerboard square slides **10 pixels to the right** over a static background.

So the true flow is `(10, 0)` on the square and `(0, 0)` everywhere else.

In [ ]:
rng = np.random.default_rng(0)

# static background: blurred random noise, so every region has some texture
background = cv2.GaussianBlur(rng.integers(0, 256, (256, 256, 3), dtype=np.uint8), (0, 0), 2)

# an 80x80 red and black checkerboard square
rows, cols = np.indices((80, 80))
square = np.zeros((80, 80, 3), dtype=np.uint8)
square[(rows // 16 + cols // 16) % 2 == 0] = (0, 0, 255)  # red in BGR

# frame 1: square at x=80, frame 2: same square at x=90
frame1 = background.copy()
frame1[88:168, 80:160] = square
frame2 = background.copy()
frame2[88:168, 90:170] = square

true_flow = np.zeros((256, 256, 2), dtype=np.float32)
true_flow[88:168, 80:160] = (10, 0)

In [ ]:
# flow_to_color only draws the flow, so any pipeline works here
viewer = OpticalFlowPipeline(OpticalFlowMethod.FARNEBACK)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, image) in zip(axes, [
    ('frame 1', cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB)),
    ('frame 2', cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB)),
    ('difference between the frames', cv2.cvtColor(cv2.absdiff(frame1, frame2), cv2.COLOR_BGR2RGB)),
    ('true flow', viewer.flow_to_color(true_flow)),
]):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis('off')

# guide line at the left edge of the square in frame 1: in frame 2 the square starts 10 px to its right
for ax in axes[:2]:
    ax.axvline(80, color='yellow', linewidth=1.5, linestyle='--')

The shift is small (10 px on a 256 px image), so the frames look almost identical side by side. The dashed line marks where the square starts in frame 1: in frame 2 there is a gap between the line and the square. The difference image makes it obvious: only the pixels where the square moved have changed.

## Comparing methods

The flow is drawn with a color wheel: hue encodes the direction of motion, saturation its magnitude.
Accuracy is measured with the **endpoint error** (EPE): the distance in pixels between the estimated and the true displacement, averaged over the image.

In [ ]:
flows = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, method in zip(axes, OpticalFlowMethod):
    pipeline = OpticalFlowPipeline(method)
    flows[method] = pipeline.run_pipeline(frame1, frame2)
    endpoint_error = np.linalg.norm(flows[method] - true_flow, axis=2).mean()

    ax.imshow(pipeline.flow_to_color(flows[method]))
    ax.set_title(f'{method.name}  (EPE {endpoint_error:.2f} px)')
    ax.axis('off')

## Warping frame 2 onto frame 1

Warping the second frame with the flow moves every pixel back to where it was in the first frame, so the square slides back to x=80 and lines up with frame 1.

The only mismatch is a thin strip right of the square. That background is visible in frame 1 but covered by the square in frame 2 (it is **occluded**), so there is nothing correct to copy from frame 2 and it picks up a piece of the square instead. This happens with any flow method, even with the true flow.

In [ ]:
pipeline = OpticalFlowPipeline(OpticalFlowMethod.RAFT_LARGE)
warped = pipeline.warp(frame2, flows[OpticalFlowMethod.RAFT_LARGE])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, image) in zip(axes, [
    ('frame 1', frame1),
    ('frame 2 warped onto frame 1', warped),
    ('difference with frame 1', cv2.absdiff(frame1, warped)),
]):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')